In [3]:
#task1
import chess
import random

def evaluate_board(board):
    
    piece_values = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0
    }

    score = 0
    for piece in board.piece_map().values():
        value = piece_values[piece.piece_type]
        score += value if piece.color == chess.WHITE else -value

    return score


def beam_search(board, beam_width=3, depth=2):
    states = [(board, [], evaluate_board(board))]

    for _ in range(depth):
        new_states = []

        for b, path, _ in states:
            for move in b.legal_moves:
                new_board = b.copy()
                new_board.push(move)

                score = evaluate_board(new_board)
                new_states.append((new_board, path + [move], score))

        # keep best beam_width states
        new_states.sort(key=lambda x: x[2], reverse=True)
        states = new_states[:beam_width]

    best_state = max(states, key=lambda x: x[2])
    return best_state[1], best_state[2]


board = chess.Board()

moves, score = beam_search(board, beam_width=3, depth=2)

print("Best move sequence:", moves)
print("Evaluation score:", score)

Best move sequence: [Move.from_uci('g1h3'), Move.from_uci('g8h6')]
Evaluation score: 0


In [6]:
#task2
import random
import math

def distance(p1, p2):
    return math.dist(p1, p2)

def total_distance(route, points):
    dist = 0
    for i in range(len(route)-1):
        dist += distance(points[route[i]], points[route[i+1]])
    return dist

def hill_climb(points):
    n = len(points)
    route = list(range(n))
    random.shuffle(route)

    best_distance = total_distance(route, points)

    improved = True
    while improved:
        improved = False

        for i in range(n):
            for j in range(i+1, n):
                new_route = route[:]
                new_route[i], new_route[j] = new_route[j], new_route[i]

                new_distance = total_distance(new_route, points)

                if new_distance < best_distance:
                    route = new_route
                    best_distance = new_distance
                    improved = True

    return route, best_distance


points = [(2,3),(5,4),(1,7),(8,2),(6,6)]

route, dist = hill_climb(points)

print("Optimized route:", route)
print("Total distance:", dist)

Optimized route: [2, 0, 1, 4, 3]
Total distance: 13.993587218285409


In [7]:
import random
import math

POP_SIZE = 100
GENERATIONS = 200
MUTATION_RATE = 0.1

cities = [(random.randint(0,50), random.randint(0,50)) for _ in range(10)]

def distance(a,b):
    return math.dist(a,b)

def route_distance(route):
    dist = 0
    for i in range(len(route)-1):
        dist += distance(cities[route[i]], cities[route[i+1]])
    return dist

def create_population():
    pop = []
    base = list(range(len(cities)))
    for _ in range(POP_SIZE):
        random.shuffle(base)
        pop.append(base[:])
    return pop

def crossover(p1,p2):
    start,end = sorted(random.sample(range(len(p1)),2))
    child = [-1]*len(p1)
    child[start:end] = p1[start:end]

    pointer = 0
    for gene in p2:
        if gene not in child:
            while child[pointer] != -1:
                pointer +=1
            child[pointer] = gene
    return child

def mutate(route):
    if random.random() < MUTATION_RATE:
        i,j = random.sample(range(len(route)),2)
        route[i],route[j] = route[j],route[i]

def genetic_algorithm():
    population = create_population()

    for _ in range(GENERATIONS):
        population.sort(key=route_distance)
        new_pop = population[:20]

        while len(new_pop) < POP_SIZE:
            p1,p2 = random.sample(population[:50],2)
            child = crossover(p1,p2)
            mutate(child)
            new_pop.append(child)

        population = new_pop

    best = min(population,key=route_distance)
    return best, route_distance(best)

best_route, dist = genetic_algorithm()

print("Best Route:", best_route)
print("Distance:", dist)

Best Route: [7, 9, 5, 4, 8, 3, 0, 1, 2, 6]
Distance: 109.68072490139275


In [10]:
#task4

import heapq
def evaluate(loads):
    return max(loads)

def beam_task_allocation(tasks,processors,beam_width=2):
    states = [([],[0]*processors)]

    for task_time,priority in tasks:
        new_states=[]
        for assignment,loads in states:
            for p in range(processors):
                new_assignment=assignment +[p]
                new_loads=loads[:]
                new_loads[p]+=task_time

                new_states.append((new_assignment,new_loads))
        new_states.sort(key=lambda x: evaluate(x[1]))
        states=new_states[:beam_width]
    best = min(states,key=lambda x: evaluate(x[1]))
    return best

tasks = [(5,1),(3,2),(6,1),(2,3),(4,2)]  

assignment, loads = beam_task_allocation(tasks, processors=3)

print("Task allocation:", assignment)
print("Processor loads:", loads)
print("Max load:", max(loads))

Task allocation: [0, 1, 2, 1, 0]
Processor loads: [9, 5, 6]
Max load: 9
